In [ ]:
!git clone https://github.com/miezansam2023-ctrl/mobile-money-pipeline-ci.git
%cd mobile-money-pipeline-ci

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **E — Extraction**

In [ ]:
# Installer SQLAlchemy et psycopg2
!pip install sqlalchemy psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 59.8 MB/s eta 0:00:00


Importation des blibliothèques

In [7]:
import pandas as pd # # manipulation de tableaux de données (DataFrame)
import numpy as np # calculs numériques
import sqlalchemy # ORM Python — connexion aux bases de données
import psycopg2 # driver PostgreSQL pour Python
import json # format JSON
import os # opérations fichiers
import time # mesure du temps
import warnings
from datetime import datetime
from sqlalchemy import create_engine, text
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None) # afficher toutes les colonnes
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

Connexion à supabase

In [8]:
# CONNEXION SUPABASE

# Importer l'URL de connexion
from google.colab import userdata
SUPABASE_URL = userdata.get('CONNEXION_POSTGRE')

# Tester la connexion
try: # try (execution du code)/except(explique pq le code ne marche ps si cela survient) = tenter un bloc de code et attraper les erreurs
    engine = sqlalchemy.create_engine(SUPABASE_URL) # crée le moteur de connexion SQLAlchemy
    with engine.connect() as conn: # ouvre une connexion temporaire
        result = conn.execute(sqlalchemy.text('SELECT version()')) # exécute une requête SQL simple
        version = result.fetchone()[0] # .fetchone()[0] = première valeur de la première ligne
        print(f'✅ Connexion Supabase réussie !')
        print(f' PostgreSQL : {version[:50]}...') # affiche les 50 premiers caractères de la version
except Exception as e: # si une erreur survient...
    print(f'❌ Erreur de connexion : {e}') # on affiche le message d'erreur
    print('Vérifiez votre SUPABASE_URL et que le projet est actif.')

✅ Connexion Supabase réussie !
 PostgreSQL : PostgreSQL 17.6 on aarch64-unknown-linux-gnu, comp...


Chargement du dataset

In [10]:
debut = time.time() #prendre le temps à l'instant t
df_raw = pd.read_csv('/content/drive/MyDrive/Projet_Data_Engineering_AG_MS/code_&_donnees/transactions_mobile_money_100k.csv', # chemin du fichier uploadé dans Colab
 encoding='utf-8', # encodage pour les accents (é, è, ê...)
 low_memory=False # évite les avertissements sur les types mixtes
)
duree = time.time() - debut
taille = df_raw.memory_usage(deep=True).sum() / 1024**2

print(f"✅ Fichier chargé avec succès.")
print(f"   Lignes   : {df_raw.shape[0]:,} en {duree:.2f}s")
print(f"   Colonnes : {df_raw.shape[1]}")
print(f"   Mémoire  : {taille:.1f} Mo")

✅ Fichier chargé avec succès.
   Lignes   : 100,000 en 1.10s
   Colonnes : 12
   Mémoire  : 61.6 Mo


Premier aperçu des données

In [ ]:
# Les 5 premières lignes
print("Aperçu des 5 premières lignes des données")
display(df_raw.head())

In [ ]:
# Types de chaque colonne
print("\n Types de données des colonnes de notre data frame : ")
print(df_raw.dtypes)

Valeurs uniques des colonnes

In [ ]:
print("=== Valeurs uniques ===")
for col in ['operateur', 'type_operation', 'statut', 'zone_expediteur']:
    valeurs = df_raw[col].unique()
    print(f"\n{col} ({len(valeurs)} valeurs) :")
    print(valeurs)

Audit des valeurs manquantes

In [ ]:
manquants = df_raw.replace('', np.nan).isnull().sum()
pct       = (manquants / len(df_raw) * 100).round(2)

audit = pd.DataFrame({
    'Nb manquants' : manquants,
    'Pourcentage %': pct
})

print("=== Valeurs manquantes ===")
print(audit[audit['Nb manquants'] > 0])
print(f"\nTotal des valeurs manquants : {manquants.sum():,}")

Statistiques descriptives

In [ ]:
print("=== Statistiques sur les montants ===")
print(df_raw[['montant_fcfa', 'frais_fcfa']].describe().round(0))

# **T — Transformation (nettoyage)**

Copie et correction des types

In [ ]:
df = df_raw.copy()

In [ ]:
# Convertir date_heure de texte vers datetime
df['date_heure'] = pd.to_datetime(df['date_heure']) # pd.to_datetime() interprète 'AAAA-MM-JJ HH:MM:SS'

# Extraire les composantes temporelles utiles
df['heure'] = df['date_heure'].dt.hour # .dt.hour = heure (0 à 23)

df['jour_semaine'] = df['date_heure'].dt.day_name() # .dt.day_name() = nom du jour en anglais

df['mois'] = df['date_heure'].dt.strftime('%Y-%m') # .strftime('%Y-%m')= '2024-01', '2024-02'...

print(' Type date_heure :', df['date_heure'].dtype)

print(df[['date_heure','heure','jour_semaine','mois']].head(3).to_string())

Nettoyages des valeurs manquantes et aberrations

In [ ]:
# Uniformiser les chaînes vides en NaN
df = df.replace('', np.nan) # les '' et NaN sont maintenant unifiés

# frais_fcfa manquants → 0 (recharges sans frais)
df['frais_fcfa'] = df['frais_fcfa'].fillna(0).astype(int) # .fillna(0) remplace NaN par 0 | .astype(int) convertit float → entier

# zone_beneficiaire manquante → 'Zone inconnue'
df['zone_beneficiaire'] = df['zone_beneficiaire'].fillna('Zone inconnue')

# id_agent manquant → 'AGT-INCONNU'
df['id_agent'] = df['id_agent'].fillna('AGT-INCONNU')

# Filtrer les montants aberrants (≤ 0)
nb_avant = len(df)
aberrants = df[df['montant_fcfa'] <= 0].copy() # sauvegarde des aberrations pour la traçabilité
df = df[df['montant_fcfa'] > 0].copy() # on ne garde que les montants strictement positifs

print(f'Aberrations supprimées : {len(aberrants)} ({len(aberrants)/nb_avant*100:.1f}%)')
print(f'Valeurs manquantes restantes : {df.isnull().sum().sum()}') # double.sum() = total global
print(f'Lignes conservées : {len(df):,}')
print('Nettoyage terminé')


# **T — Transformation (enrichissement et agrégations)**

les colonnes calculées enrichies

In [ ]:
# Montant net reçu par le bénéficiaire
df['montant_net_fcfa'] = df['montant_fcfa'] - df['frais_fcfa']

# Taux de frais en pourcentage
df['taux_frais_pct'] = (df['frais_fcfa'] / df['montant_fcfa'] * 100).round(2)

# Catégorie de montant
bins = [0, 5000, 50000, 200000, float('inf')] # float('inf') = infini (pas de borne supérieure)
labels = ['Micro (≤5k)', 'Petit (5k-50k)', 'Moyen (50k-200k)', 'Gros (>200k)']
df['categorie_montant'] = pd.cut(df['montant_fcfa'], bins=bins, labels=labels, right=True) # pd.cut() découpe en intervalles automatiquement
df['categorie_montant'] = df['categorie_montant'].astype(str) # convertir en texte pour Supabase

# Tranche horaire
def tranche(h):
 if 6<=h<9: return 'Matin (6h-9h)'
 elif 9<=h<12: return 'Matinée (9h-12h)'
 elif 12<=h<14:return 'Déjeuner (12h-14h)'
 elif 14<=h<18:return 'Après-midi (14h-18h)'
 elif 18<=h<21:return 'Soirée (18h-21h)'
 else: return 'Nuit (21h-6h)'

df['tranche_horaire'] = df['heure'].apply(tranche) # .apply(fonction) = applique la fonction ligne par ligne

# Transaction inter-villes (1 si différentes, 0 si même ville)
df['inter_ville'] = (df['zone_expediteur'] != df['zone_beneficiaire']).astype(int) # True/False → 1/0
print(f'Colonnes totales après enrichissement : {len(df.columns)}')

nouvelles = ['montant_net_fcfa', 'taux_frais_pct', 'categorie_montant', 'tranche_horaire', 'inter_ville']
print(f'{len(nouvelles)} Nouvelles colonnes :')

for col in nouvelles:
    print(f"  + {col} : {df[col].dtype}")

In [ ]:
df.info()

In [ ]:
print("=== Tests de qualité ===\n")

tests = {
    "IDs uniques"         : df['id_transaction'].is_unique,
    "Montants positifs"   : (df['montant_fcfa'] > 0).all(),
    "Opérateurs valides"  : df['operateur'].isin(
                                ['MTN CI','Orange Money','Moov Africa','Wave']).all(),
    "Statuts valides"     : df['statut'].isin(
                                ['Succès','Échec','En attente']).all(),
    "Dates sans NaN"      : df['date_heure'].notna().all(),
}

tous_ok = True
for nom, resultat in tests.items():
    icone = " OK!" if resultat else "❌"
    print(f"  {icone} {nom}")
    if not resultat:
        tous_ok = False

print(f"\n{'Tous les tests passent coorectement!' if tous_ok else '❌ Corriger les tests en rouge'}")

Agrégations par opérateur, type et zone pour les transactions réussies uniquement

In [ ]:
df_succes = df[df['statut'] == 'Succès'].copy() # filtre uniquement les transactions abouties (~80%)

# Agrégation 1 — Par opérateur
agg_op = df_succes.groupby('operateur').agg( # .agg() = calculer plusieurs métriques en même temps
 nb_transactions = ('id_transaction','count'), # count = nombre de lignes
 volume_fcfa = ('montant_fcfa','sum'), # sum = somme
 montant_moyen = ('montant_fcfa','mean'), # mean = moyenne
 montant_median = ('montant_fcfa','median'), # median = médiane
 frais_total = ('frais_fcfa','sum'),
 ).round(0).reset_index()

agg_op.columns = ['operateur','nb_transactions','volume_fcfa','montant_moyen','montant_median','frais_total']   # renommer proprement

agg_op['part_volume_pct'] = (agg_op['volume_fcfa']/agg_op['volume_fcfa'].sum()*100).round(1)

agg_op = agg_op.astype({'nb_transactions':int,'volume_fcfa':int,'montant_moyen':int,'montant_median':int,'frais_total':int}) # convertir en entiers pour PostgreSQL

# Agrégation 2 — Par type d'opération
agg_type = df_succes.groupby('type_operation').agg(
 nb = ('id_transaction','count'),
 volume = ('montant_fcfa','sum'),
 moy = ('montant_fcfa','mean'),
 taux_frais = ('taux_frais_pct','mean'),
).round(0).sort_values('volume',ascending=False).reset_index()

agg_type.columns = ['type_operation','nb','volume','moy','taux_frais']

agg_type = agg_type.astype({'nb':int,'volume':int,'moy':int})

# Agrégation 3 — Par zone

agg_zone = df_succes.groupby('zone_expediteur')['montant_fcfa'].sum().sort_values(ascending=False).reset_index()

agg_zone.columns = ['zone','volume_fcfa']

agg_zone['part_pct'] = (agg_zone['volume_fcfa']/agg_zone['volume_fcfa'].sum()*100).round(1)

agg_zone = agg_zone.astype({'volume_fcfa':int})

print(f' {len(df_succes):,} transactions réussies analysées')
print(f'Volume total : {df_succes["montant_fcfa"].sum():,} FCFA')

# **L — Chargement dans Supabase**

Chargement de la table principale transactions

In [ ]:
# Préparer le DataFrame pour PostgreSQL
df_load = df.copy()
df_load['date_heure'] = df_load['date_heure'].astype(str) # convertir datetime → texte pour éviter les problèmes de timezone

# Paramètres de chargement par lots
CHUNK_SIZE = 5000 # nombre de lignes par lot
TOTAL = len(df_load)
chunks = [df_load[i:i+CHUNK_SIZE] for i in range(0, TOTAL, CHUNK_SIZE)]
print(f'Chargement de {TOTAL:,} lignes en {len(chunks)} lots de {CHUNK_SIZE}...')

debut_load = time.time()
lignes_chargees = 0 # compteur de lignes chargées

with engine.connect() as conn: # ouvre la connexion Supabase
    for i, chunk in enumerate(chunks): # on parcourt chaque lot
        chunk.to_sql(
            'transactions',
            conn,
            if_exists='append',
            index=False,
            method='multi',
            chunksize=5000
        )

        lignes_chargees += len(chunk)
        pct = (lignes_chargees / TOTAL) * 100
        print(f' Lot {i+1:2d}/{len(chunks)} — {lignes_chargees:,}/{TOTAL:,} lignes ({pct:.0f}%)', end='\r')
        conn.commit() # .commit() = valider toutes les insertions à chaque lot

duree_load = time.time() - debut_load
print(f'\n✅ Table transactions chargée : {lignes_chargees:,} lignes en {duree_load:.1f}s')

Chargement des tables d'aggregation

In [ ]:
tables_agg = [ # liste des agrégations à charger
 ('agg_operateur', agg_op), # nom de la table Supabase, DataFrame correspondant
 ('agg_type_operation', agg_type),
 ('agg_zone', agg_zone),
]
with engine.connect() as conn:
 for nom_table, df_agg in tables_agg:
      df_agg.to_sql(nom_table, conn,
                    if_exists='replace', # replace = vider et recréer (tables petites → pas de problème)
                    index=False, method='multi')
      print(f' ✅ {nom_table} : {len(df_agg)} lignes')
conn.commit()
print('\n✅ Toutes les tables ont été chargées dans Supabase.')

Export Parquet

In [ ]:
df_export = df.copy()

df_export['date_heure'] = df_export['date_heure'].astype(str)

df_export.to_parquet('transactions.parquet',
 engine='pyarrow', compression='snappy', index=False) # snappy = compression rapide, bon ratio

taille_csv = os.path.getsize('/content/transactions_mobile_money_100k.csv') / 1024**2

taille_pq = os.path.getsize('transactions.parquet') / 1024**2

print(f'CSV : {taille_csv:.1f} Mo')
print(f'Parquet : {taille_pq:.1f} Mo')
print(f'Gain : {(1-taille_pq/taille_csv)*100:.0f}% de compression')

Rapport final

In [ ]:
# RAPPORT DE PIPELINE ETL

sep = '=' * 62
print(sep)
print(' PIPELINE ETL — MOBILE MONEY CÔTE D\'IVOIRE')
print(f' Généré le : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(sep)
print('\n EXTRACT')
print(f' Fichier : transactions_mobile_money_100k.csv')
print(f' Lignes : {100000:,}')
print('\n TRANSFORM')
print(f' date_heure → datetime64 : ✅')
print(f' frais NaN → 0 (2 000 cas) : ✅')
print(f' zone NaN → Zone inconnue : ✅')
print(f' agent NaN → AGT-INCONNU : ✅')
print(f' Aberrations filtrées : 200 : ✅')
print(f' Colonnes enrichies : +5 : ✅')
print(f' Lignes propres : {len(df):,}')
print('\n LOAD')
print(f' transactions : {len(df):,} lignes ✅')
print(f' agg_operateur : {len(agg_op)} lignes ✅')
print(f' agg_type_operation : {len(agg_type)} lignes ✅')
print(f' agg_zone : {len(agg_zone)} lignes ✅')
print(f' Parquet : {taille_pq:.1f} Mo (gain 77%) ✅')
print('\n KPI FINAUX')
print(f' Volume total : {df_succes["montant_fcfa"].sum():>15,} FCFA')
print(f' Frais totaux : {df_succes["frais_fcfa"].sum():>15,} FCFA')
print(f' Taux succès : {len(df_succes)/len(df)*100:>14.1f} %')
print(f' Montant moyen : {df_succes["montant_fcfa"].mean():>15,.0f} FCFA')
top_op = agg_op.sort_values('volume_fcfa',ascending=False).iloc[0]
print(f'\n 🏆 Opérateur dominant : {top_op["operateur"]} ({top_op["part_volume_pct"]}%)')
print(f' 🏆 Type dominant : {agg_type.iloc[0]["type_operation"]}')
print(f' 🏆 Zone dominante : {agg_zone.iloc[0]["zone"]}')
engine.dispose() # .dispose() = ferme toutes les connexions du pool — équivalent de conn.close()
print('\n✅ Connexion Supabase fermée. Pipeline ETL terminé.')

# **Peupler les tables de dimensions**

In [ ]:
!pip install sqlalchemy psycopg2-binary -q

dim_operateur

In [ ]:
# -- dim_operateur --
dim_op = pd.DataFrame({
    'nom_operateur':  ['MTN CI','Orange Money','Moov Africa','Wave'],
    'type_operateur': ['Telecoms','Telecoms','Telecoms','Fintech'],
    'actif':          [True, True, True, True]
})
with engine.connect() as conn:
    conn.execute(text('DELETE FROM dim_operateur'))   # vider avant de remplir
    conn.execute(text('ALTER SEQUENCE dim_operateur_id_operateur_seq RESTART WITH 1'))   # remettre le compteur a 1
    dim_op.to_sql('dim_operateur', conn, if_exists='append', index=False, method='multi')
    conn.commit()
print(f'dim_operateur : {len(dim_op)} lignes ✅')

dim_zone

In [ ]:
# -- dim_zone --
infos = {
    'Abidjan-Cocody':   ('Abidjan','Quartier Abidjan',600000),
    'Abidjan-Plateau':  ('Abidjan','Quartier Abidjan',50000),
    'Abidjan-Yopougon':('Abidjan','Quartier Abidjan',1200000),
    'Abidjan-Adjame':   ('Abidjan','Quartier Abidjan',350000),
    'Abidjan-Marcory':  ('Abidjan','Quartier Abidjan',280000),
    'Bouake':           ('Centre','Ville interieur',800000),
    'Korhogo':          ('Nord','Ville interieur',400000),
    'Yamoussoukro':     ('Centre','Capitale politique',300000),
    'San Pedro':        ('Sud-Ouest','Ville portuaire',250000),
    'Man':              ('Ouest','Ville interieur',200000),
    'Zone inconnue':    ('Inconnu','Non identifiee',0),
}

toutes_zones = pd.concat([df['zone_expediteur'],df['zone_beneficiaire']]).dropna().unique()

rows = [{'nom_zone':z,'region':infos.get(z,('Inconnu','Autre',0))[0],
         'type_zone':infos.get(z,('Inconnu','Autre',0))[1],
         'population':infos.get(z,('Inconnu','Autre',0))[2]
         }
        for z in sorted(toutes_zones)]

dim_z = pd.DataFrame(rows)

with engine.connect() as conn:
    conn.execute(text('DELETE FROM dim_zone'))
    conn.execute(text('ALTER SEQUENCE dim_zone_id_zone_seq RESTART WITH 1'))
    dim_z.to_sql('dim_zone', conn, if_exists='append', index=False,
method='multi')
    conn.commit()

print(f'dim_zone : {len(dim_z)} lignes ✅')

dim_type_operation

In [ ]:
 # -- dim_type_operation --
dim_type = pd.DataFrame({
    'nom_type':['Transfert','Depot','Retrait','Paiement marchand','Recharge telephone'],
    'categorie':['Envoi de fonds','Entree de fonds','Sortie de fonds','Paiement','Service'],
    'taux_frais_max':[1.0, 0.7, 0.8, 1.0, 0.0]
})
with engine.connect() as conn:
    conn.execute(text('DELETE FROM dim_type_operation'))
    conn.execute(text('ALTER SEQUENCE dim_type_operation_id_type_seq RESTART WITH 1'))
    dim_type.to_sql('dim_type_operation', conn, if_exists='append', index=False,
method='multi')
    conn.commit()
print(f'dim_type_operation : {len(dim_type)} lignes ✅')

dim_date

In [ ]:
# -- dim_date --
dates = pd.date_range('2024-01-01','2024-06-30',freq='D')   # pd.date_range = serie de dates quotidiennes
noms_mois = {1:'Janvier',2:'Fevrier',3:'Mars',4:'Avril',5:'Mai',6:'Juin'}
noms_jours =  {0:'Lundi',1:'Mardi',2:'Mercredi',3:'Jeudi',4:'Vendredi',5:'Samedi',6:'Dimanche'}

dim_d = pd.DataFrame({
    'date_complete': dates.date,   # .date = seulement la date, sans l'heure
    'annee': dates.year, 'mois': dates.month,
    'nom_mois': dates.month.map(noms_mois),   # .map(dico) = remplacer chaque numero par son nom
    'trimestre': dates.quarter, 'semaine': dates.isocalendar().week.astype(int),
    'jour': dates.day,
    'nom_jour': dates.dayofweek.map(noms_jours),   # 0=Lundi, 6=Dimanche
    'est_weekend': dates.dayofweek >= 5   # True si samedi(5) ou dimanche(6)
})

with engine.connect() as conn:
    conn.execute(text('DELETE FROM dim_date'))
    conn.execute(text('ALTER SEQUENCE dim_date_id_date_seq RESTART WITH 1'))
    dim_d.to_sql('dim_date', conn, if_exists='append', index=False,
method='multi')
    conn.commit()

print(f'dim_date : {len(dim_d)} lignes ✅')

# **Peupler la table de faits**

In [ ]:
# -- Charger les IDs des dimensions depuis Supabase --
map_op   = dict(zip(*pd.read_sql('SELECT nom_operateur, id_operateur FROM dim_operateur', engine).values.T))

map_zone = dict(zip(*pd.read_sql('SELECT nom_zone, id_zone FROM dim_zone', engine).values.T))

map_type = dict(zip(*pd.read_sql('SELECT nom_type, id_type FROM dim_type_operation', engine).values.T))

tmp = pd.read_sql('SELECT id_date, date_complete FROM dim_date', engine)

tmp['date_complete'] = pd.to_datetime(tmp['date_complete']).dt.date

map_date = dict(zip(tmp['date_complete'], tmp['id_date']))

print('Lookups ✅ :', len(map_op), 'operateurs,', len(map_zone), 'zones,',

len(map_date), 'dates')

In [ ]:
df['date'] = df['date_heure'].dt.date

faits = pd.DataFrame({
    'id_transaction':  df['id_transaction'],
    'id_operateur':    df['operateur'].map(map_op),   # .map() remplace 'MTN CI' par 1, 'Wave' par 4...
    'id_zone_exp':     df['zone_expediteur'].map(map_zone),
    'id_zone_ben':     df['zone_beneficiaire'].map(map_zone),
    'id_type':         df['type_operation'].map(map_type),
    'id_date':         df['date'].map(map_date),
    'montant_fcfa':    df['montant_fcfa'],
    'frais_fcfa':      df['frais_fcfa'],
    'montant_net_fcfa':df['montant_net_fcfa'],
    'taux_frais_pct':  df['taux_frais_pct'],
    'heure':           df['heure'],
    'statut':          df['statut'],
    'inter_ville':     df['inter_ville'],
    'id_agent':        df['id_agent'],
})

# -- Supprimer les lignes avec FK manquante --
cles = ['id_operateur','id_zone_exp','id_zone_ben','id_type','id_date']

faits = faits.dropna(subset=cles)   # .dropna(subset) = supprimer les lignes avec NaN dans ces colonnes

faits[cles] = faits[cles].astype(int)   # convertir en entiers pour PostgreSQL

print(f'Lignes a charger : {len(faits):,} ✅')

In [ ]:
 # -- Chargement par lots de 5000 --
import time
debut = time.time()
chunks = [faits[i:i+5000] for i in range(0, len(faits), 5000)]   # decouper en lots
with engine.connect() as conn:
    conn.execute(text('DELETE FROM faits_transactions'))
    for i, chunk in enumerate(chunks):
        chunk.to_sql('faits_transactions', conn, if_exists='append', index=False,
method='multi', chunksize=500)
        print(f'  Lot {i+1}/{len(chunks)}... ✅', end='\r')
    conn.commit()
print(f'\n faits_transactions : {len(faits):,} lignes en {time.time() - debut:.0f}s ✅')

# **Requetes analytiques SQL**

Requête 1 : parts de marché par opérateur

In [ ]:
# SELECT
#     o.nom_operateur,
#     COUNT(*)                                               AS nb_transactions,
#     ROUND(SUM(f.montant_fcfa) / 1e6, 2)                   AS volume_millions_fcfa,
#     ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS part_marche_pct
# FROM faits_transactions f
# JOIN dim_operateur o ON f.id_operateur = o.id_operateur
# WHERE f.statut = 'Succès'
# GROUP BY o.nom_operateur
# ORDER BY nb_transactions DESC;


| nom_operateur | nb_transactions | volume_millions_fcfa | part_marche_pct |
| ------------- | --------------- | -------------------- | --------------- |
| Wave          | 15121           | 2156.04              | 25.19           |
| Orange Money  | 15058           | 2102.88              | 25.08           |
| Moov Africa   | 15005           | 2117.91              | 24.99           |
| MTN CI        | 14849           | 2119.02              | 24.73           |

Requête 2 : tendances mensuelles

In [ ]:
# SELECT
#     d.mois,
#     d.nom_mois,
#     COUNT(*)                        AS nb_transactions,
#     ROUND(SUM(f.montant_fcfa)/1e6, 2) AS volume_millions_fcfa,
#     ROUND(AVG(f.montant_fcfa), 0)   AS montant_moyen_fcfa
# FROM faits_transactions f
# JOIN dim_date d ON f.id_date = d.id_date
# GROUP BY d.mois, d.nom_mois
# ORDER BY d.mois;

| mois | nom_mois | nb_transactions | volume_millions_fcfa | montant_moyen_fcfa |
| ---- | -------- | --------------- | -------------------- | ------------------ |
| 1    | Janvier  | 10209           | 1442.00              | 141248             |
| 2    | Fevrier  | 9593            | 1374.17              | 143247             |
| 3    | Mars     | 10337           | 1469.13              | 142124             |
| 4    | Avril    | 9870            | 1388.00              | 140628             |
| 5    | Mai      | 10449           | 1479.32              | 141575             |
| 6    | Juin     | 9575            | 1343.22              | 140284             |

Requête 3 : taux d'échec par opérateur



In [ ]:
# SELECT
#     operateur,
#     COUNT(*)                                                        AS total,
#     SUM(CASE WHEN statut = 'Échec' THEN 1 ELSE 0 END)             AS nb_echecs,
#     ROUND(100.0 * SUM(CASE WHEN statut = 'Échec' THEN 1 ELSE 0 END)
#           / COUNT(*), 2)                                            AS taux_echec_pct
# FROM transactions
# GROUP BY operateur
# ORDER BY taux_echec_pct DESC;

| operateur    | total | nb_echecs | taux_echec_pct |
| ------------ | ----- | --------- | -------------- |
| Moov Africa  | 25036 | 2516      | 10.05          |
| Orange Money | 25047 | 2496      | 9.97           |
| MTN CI       | 24639 | 2419      | 9.82           |
| Wave         | 25078 | 2433      | 9.70           |

Requête 4 : Classement des opérateurs par mois

In [ ]:
# -- Requête 4 : Classement des opérateurs par mois (CTE + RANK)
# WITH volume_mensuel AS (
#     SELECT
#         d.mois,
#         d.nom_mois,
#         o.nom_operateur,
#         SUM(f.montant_fcfa)          AS volume_fcfa,
#         COUNT(*)                     AS nb_transactions
#     FROM faits_transactions f
#     JOIN dim_date d       ON f.id_date      = d.id_date
#     JOIN dim_operateur o  ON f.id_operateur = o.id_operateur
#     GROUP BY d.mois, d.nom_mois, o.nom_operateur
# )
# SELECT
#     mois,
#     nom_mois,
#     nom_operateur,
#     volume_fcfa,
#     nb_transactions,
#     RANK() OVER (
#         PARTITION BY mois
#         ORDER BY volume_fcfa DESC
#     ) AS rang_mensuel
# FROM volume_mensuel
# ORDER BY mois, rang_mensuel;

| mois | nom_mois | nom_operateur | volume_fcfa | nb_transactions | rang_mensuel |
| ---- | -------- | ------------- | ----------- | --------------- | ------------ |
| 1    | Janvier  | MTN CI        | 368648740   | 2546            | 1            |
| 1    | Janvier  | Moov Africa   | 359952592   | 2519            | 2            |
| 1    | Janvier  | Wave          | 356999851   | 2562            | 3            |
| 1    | Janvier  | Orange Money  | 356402159   | 2582            | 4            |
| 2    | Fevrier  | Orange Money  | 346997258   | 2420            | 1            |
| 2    | Fevrier  | Wave          | 345647762   | 2400            | 2            |
| 2    | Fevrier  | MTN CI        | 344049670   | 2382            | 3            |
| 2    | Fevrier  | Moov Africa   | 337473539   | 2391            | 4            |
| 3    | Mars     | Wave          | 389449738   | 2611            | 1            |
| 3    | Mars     | Moov Africa   | 377464043   | 2676            | 2            |
| 3    | Mars     | MTN CI        | 352562273   | 2499            | 3            |
| 3    | Mars     | Orange Money  | 349658490   | 2551            | 4            |
| 4    | Avril    | Moov Africa   | 353367739   | 2465            | 1            |
| 4    | Avril    | MTN CI        | 351907965   | 2484            | 2            |
| 4    | Avril    | Orange Money  | 343984236   | 2457            | 3            |
| 4    | Avril    | Wave          | 338738755   | 2464            | 4            |
| 5    | Mai      | Wave          | 378098283   | 2640            | 1            |
| 5    | Mai      | Moov Africa   | 369431995   | 2625            | 2            |
| 5    | Mai      | Orange Money  | 367138418   | 2652            | 3            |
| 5    | Mai      | MTN CI        | 364652452   | 2532            | 4            |
| 6    | Juin     | Wave          | 347105078   | 2444            | 1            |
| 6    | Juin     | Orange Money  | 338695964   | 2396            | 2            |
| 6    | Juin     | MTN CI        | 337194081   | 2406            | 3            |
| 6    | Juin     | Moov Africa   | 320221778   | 2329            | 4            |

# **Tableau de bord Matplotlib**

In [ ]:
import matplotlib.pyplot as plt # création de graphiques
import matplotlib.ticker as mtick

# Configuration globale
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         10,
    'axes.titlesize':    12,
    'axes.titleweight': 'bold',
})

# Couleurs officielles des opérateurs
PALETTE = {
    'MTN CI':       '#FFCC00',
    'Orange Money': '#FF6600',
    'Moov Africa':  '#009900',
    'Wave':         '#1A73E8',
}

print("✅ Configuration graphique prête")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Tableau de bord — Mobile Money Côte d'Ivoire 2024",
             fontsize=16, fontweight='bold')

# ── GRAPHIQUE 1 : Volume par opérateur (barres) ──────
vol_op = df_succes.groupby('operateur')['montant_fcfa'].sum() / 1e9
couleurs = [PALETTE.get(op, '#999999') for op in vol_op.index]
barres = axes[0,0].bar(vol_op.index, vol_op.values, color=couleurs, edgecolor='white')
for b in barres:
    axes[0,0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
                   f'{b.get_height():.1f}B', ha='center', fontweight='bold', fontsize=9)
axes[0,0].set_title("Volume total par opérateur (Milliards FCFA)")
axes[0,0].set_ylabel("Milliards FCFA")
axes[0,0].tick_params(axis='x', rotation=15)

# ── GRAPHIQUE 2 : Parts de marché (camembert) ──
parts = df_succes.groupby('operateur')['montant_fcfa'].sum()
couleurs2 = [PALETTE.get(op, '#999999') for op in parts.index]
axes[0,1].pie(parts.values, labels=parts.index, autopct='%1.1f%%',
              colors=couleurs2, explode=[0.04]*4, startangle=90)
axes[0,1].set_title("Parts de marché par volume")

# ── GRAPHIQUE 3 : Volume par heure (courbe) ──
vol_heure = df_succes.groupby('heure')['montant_fcfa'].sum() / 1e6
axes[0,2].fill_between(vol_heure.index, vol_heure.values, alpha=0.3, color='#1565C0')
axes[0,2].plot(vol_heure.index, vol_heure.values,
               color='#1565C0', linewidth=2.5, marker='o', markersize=4)
axes[0,2].axvspan(6, 9,   alpha=0.15, color='green',  label='Matin')
axes[0,2].axvspan(17, 20, alpha=0.15, color='orange', label='Soir')
axes[0,2].set_title("Volume par tranche horaire")
axes[0,2].set_xlabel("Heure")
axes[0,2].set_ylabel("Millions FCFA")
axes[0,2].set_xticks(range(0, 24, 2))
axes[0,2].legend(fontsize=8)

# ── GRAPHIQUE 4 : Taux d'échec par opérateur ──
echec = df[df['statut'] == 'Échec'].groupby('operateur').size()
total = df.groupby('operateur').size()
taux  = (echec / total * 100).round(2).sort_values()
couleurs3 = [PALETTE.get(op, '#999999') for op in taux.index]
barres2 = axes[1,0].barh(taux.index, taux.values, color=couleurs3, alpha=0.85)
for b, val in zip(barres2, taux.values):
    axes[1,0].text(val + 0.2, b.get_y() + b.get_height()/2,
                   f'{val:.1f}%', va='center', fontweight='bold', fontsize=9)
axes[1,0].set_title("Taux d'échec par opérateur")
axes[1,0].set_xlabel("%")
axes[1,0].axvline(taux.mean(), color='red', linestyle='--', linewidth=1.5, label='Moyenne')
axes[1,0].legend(fontsize=8)

# ── GRAPHIQUE 5 : Volume mensuel (courbe) ──
vol_mensuel = df_succes.groupby('mois')['montant_fcfa'].sum() / 1e6
axes[1,1].plot(vol_mensuel.index, vol_mensuel.values,
               color='#2E7D32', marker='s', linewidth=2.5, markersize=8)
axes[1,1].fill_between(vol_mensuel.index, vol_mensuel.values, alpha=0.15, color='#2E7D32')
for x, y in zip(vol_mensuel.index, vol_mensuel.values):
    axes[1,1].annotate(f'{y:.0f}M', xy=(x, y), xytext=(0, 10),
                       textcoords='offset points', ha='center',
                       fontsize=9, fontweight='bold')
axes[1,1].set_title("Évolution mensuelle du volume (M FCFA)")
axes[1,1].set_xlabel("Mois")
axes[1,1].set_ylabel("Millions FCFA")

# ── GRAPHIQUE 6 : Volume par type d'opération ────────
vol_type = df_succes.groupby('type_operation')['montant_fcfa'].sum() / 1e6
vol_type = vol_type.sort_values(ascending=True)
axes[1,2].barh(vol_type.index, vol_type.values, color='#1565C0', alpha=0.8)
for i, val in enumerate(vol_type.values):
    axes[1,2].text(val + 10, i, f'{val:.0f}M', va='center',
                   fontweight='bold', fontsize=9)
axes[1,2].set_title("Volume par type d'opération (M FCFA)")
axes[1,2].set_xlabel("Millions FCFA")

# ── Sauvegarde ────────────────────
plt.tight_layout()
plt.savefig('dashboard_mobile_money.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard sauvegardé — dashboard_mobile_money.png")

# **Création du DAG Airflow**

In [ ]:
import os
os.makedirs('/content/mobile-money-pipeline-ci/dags', exist_ok=True)

dag_code = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

# ── Arguments par défaut ──────────────────────────────
default_args = {
    "owner":            "mobile_money_ci",
    "depends_on_past":  False,
    "start_date":       datetime(2024, 1, 1),
    "email":            ["miezamsam@gmail.com"],
    "email_on_failure": True,
    "email_on_retry":   False,
    "retries":          2,
    "retry_delay":      timedelta(minutes=5),
}

# ── Tâche 1 : Extraction ──────────────────────────────
def extraire():
    df = pd.read_csv("/data/transactions_mobile_money_100k.csv",
                     encoding="utf-8", low_memory=False)
    df.to_parquet("/data/raw/transactions_raw.parquet", index=False)
    print(f"EXTRACT : {len(df):,} lignes extraites")

# ── Tâche 2 : Nettoyage ───────────────────────────────
def nettoyer():
    df = pd.read_parquet("/data/raw/transactions_raw.parquet")
    df = df.replace("", np.nan)
    df["frais_fcfa"]        = df["frais_fcfa"].fillna(0).astype(int)
    df["zone_beneficiaire"] = df["zone_beneficiaire"].fillna("Zone inconnue")
    df["id_agent"]          = df["id_agent"].fillna("AGT-INCONNU")
    df = df[df["montant_fcfa"] > 0].copy()
    df["date_heure"]        = pd.to_datetime(df["date_heure"])
    df["montant_net_fcfa"]  = df["montant_fcfa"] - df["frais_fcfa"]
    df["heure"]             = df["date_heure"].dt.hour
    df["mois"]              = df["date_heure"].dt.month
    df["est_succes"]        = (df["statut"] == "Succès").astype(int)
    df["inter_ville"]       = (df["zone_expediteur"] != df["zone_beneficiaire"]).astype(int)
    df.to_parquet("/data/clean/transactions_clean.parquet", index=False)
    print(f"CLEAN : {len(df):,} lignes propres")

# ── Tâche 3 : Chargement Supabase ─────────────────────
def charger():
    import sqlalchemy, os
    SUPABASE_URL = os.environ.get("SUPABASE_URL", "")
    if not SUPABASE_URL:
        raise ValueError("SUPABASE_URL non définie !")
    engine = sqlalchemy.create_engine(SUPABASE_URL)
    df = pd.read_parquet("/data/clean/transactions_clean.parquet")
    with engine.connect() as conn:
        df.to_sql("transactions_raw", conn,
                  if_exists="replace", index=False, chunksize=5000)
        conn.commit()
    engine.dispose()
    print(f"LOAD : {len(df):,} lignes chargées")

# ── Tâche 4 : Rapport JSON ────────────────────────────
def rapport():
    import json
    from datetime import datetime as dt
    df = pd.read_parquet("/data/clean/transactions_clean.parquet")
    df_s = df[df["statut"] == "Succès"]
    kpi = {
        "date_generation":    dt.now().strftime("%Y-%m-%d %H:%M"),
        "nb_total":           int(len(df)),
        "nb_succes":          int(len(df_s)),
        "taux_succes_pct":    round(len(df_s)/len(df)*100, 1),
        "volume_total_fcfa":  int(df_s["montant_fcfa"].sum()),
        "montant_moyen_fcfa": int(df_s["montant_fcfa"].mean()),
    }
    with open("/data/output/rapport_nuit.json", "w") as f:
        json.dump(kpi, f, ensure_ascii=False, indent=2)
    print("RAPPORT : rapport_nuit.json généré")

# ── Définition du DAG ─────────────────────────────────
with DAG(
    dag_id="pipeline_mobile_money_ci",
    default_args=default_args,
    description="Pipeline ETL Mobile Money CI — quotidien 23h",
    schedule="0 23 * * *",   # chaque jour à 23h
    catchup=False,
    max_active_runs=1,
    tags=["etl", "mobile_money", "cote_ivoire"],
) as dag:

    t1 = PythonOperator(task_id="extraire",  python_callable=extraire)
    t2 = PythonOperator(task_id="nettoyer",  python_callable=nettoyer)
    t3 = PythonOperator(task_id="charger",   python_callable=charger)
    t4 = PythonOperator(task_id="rapport",   python_callable=rapport)

    # Ordre d'exécution : t1 → t2 → t3 → t4
    t1 >> t2 >> t3 >> t4
'''

with open('/content/mobile-money-pipeline-ci/dags/dag_mobile_money.py', 'w') as f:
    f.write(dag_code.strip())

print("✅ DAG sauvegardé : dags/dag_mobile_money.py")

# **Création du Dockerfile**

In [ ]:
dockerfile = '''
FROM python:3.10-slim

LABEL maintainer="miezamsam@gmail.com"
LABEL description="Pipeline ETL Mobile Money Côte d Ivoire"

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

RUN apt-get update && apt-get install -y \\
    build-essential \\
    libpq-dev \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

RUN mkdir -p /data/raw /data/clean /data/output

CMD ["python", "notebooks/pipeline_etl.py"]
'''

with open('/content/mobile-money-pipeline-ci/Dockerfile', 'w') as f:
    f.write(dockerfile.strip())

print("✅ Dockerfile créé")

# **Création du docker-compose.yml**

In [ ]:
compose = '''
version: "3.8"

services:

  pipeline:
    build: .
    container_name: pipeline_mobile_money
    env_file: .env
    volumes:
      - ./data:/data
      - ./notebooks:/app/notebooks
    depends_on:
      - postgres

  postgres:
    image: postgres:15-alpine
    container_name: pipeline_postgres
    environment:
      POSTGRES_USER:     airflow
      POSTGRES_PASSWORD: airflow_secret
      POSTGRES_DB:       airflow_db
    volumes:
      - postgres_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD", "pg_isready", "-U", "airflow"]
      interval: 5s
      retries: 5

volumes:
  postgres_data:
'''

with open('/content/mobile-money-pipeline-ci/docker-compose.yml', 'w') as f:
    f.write(compose.strip())

print("✅ docker-compose.yml créé")

# **Fichier SQL schéma en étoile commenté**

In [ ]:
# import os
# os.makedirs('/content/mobile-money-pipeline-ci/sql', exist_ok=True)

# sql_content = """
# -- ============================================================
# -- FICHIER : schema_mobile_money.sql
# -- PROJET  : Pipeline ETL Mobile Money — Côte d'Ivoire
# -- AUTEUR  : Miézan Sam & Ariel Gnapié
# -- DATE    : 12 06 2026
# -- DESC    : Création complète du schéma en étoile et des
# --           tables d'agrégation pour l'analyse Mobile Money
# -- ============================================================


# -- ============================================================
# -- PARTIE 1 : TABLE BRUTE DES TRANSACTIONS
# -- Contient toutes les transactions telles que chargées
# -- depuis le CSV — avant modélisation en étoile
# -- ============================================================

# CREATE TABLE IF NOT EXISTS transactions_raw (
#     id_transaction    TEXT PRIMARY KEY,        -- identifiant unique de la transaction
#     date_heure        TIMESTAMP,               -- date et heure exacte de la transaction
#     operateur         TEXT,                    -- MTN CI, Orange Money, Moov Africa, Wave
#     type_operation    TEXT,                    -- Dépôt, Retrait, Transfert, Paiement, Recharge
#     expediteur        TEXT,                    -- numéro ou identifiant de l'expéditeur
#     beneficiaire      TEXT,                    -- numéro ou identifiant du bénéficiaire
#     montant_fcfa      BIGINT,                  -- montant de la transaction en FCFA
#     frais_fcfa        INTEGER DEFAULT 0,       -- frais prélevés par l'opérateur
#     zone_expediteur   TEXT,                    -- zone géographique de l'expéditeur
#     zone_beneficiaire TEXT,                    -- zone géographique du bénéficiaire
#     id_agent          TEXT,                    -- agent Mobile Money ayant traité l'opération
#     statut            TEXT,                    -- Succès, Échec, En attente
#     -- Colonnes enrichies (calculées lors du nettoyage ETL)
#     montant_net_fcfa  BIGINT,                  -- montant_fcfa - frais_fcfa
#     heure             INTEGER,                 -- heure extraite de date_heure (0 à 23)
#     mois              INTEGER,                 -- mois extrait de date_heure (1 à 12)
#     annee             INTEGER,                 -- année extraite de date_heure
#     jour_semaine      TEXT,                    -- nom du jour (Monday, Tuesday...)
#     est_succes        INTEGER DEFAULT 0,       -- 1 si statut = Succès, 0 sinon
#     inter_ville       INTEGER DEFAULT 0,       -- 1 si zones expéditeur ≠ bénéficiaire
#     tranche_montant   TEXT                     -- < 5k / 5k-20k / 20k-100k / > 100k
# );


# -- ============================================================
# -- PARTIE 2 : TABLES DE DIMENSIONS
# -- Les dimensions décrivent QUI, QUOI, OÙ, QUAND
# -- Elles entourent la table de faits dans le schéma en étoile
# -- ============================================================

# -- ------------------------------------------------------------
# -- DIMENSION 1 : Opérateur
# -- Décrit les 4 opérateurs Mobile Money actifs en Côte d'Ivoire
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS dim_operateur (
#     id_operateur   SERIAL PRIMARY KEY,         -- identifiant auto-incrémenté (1, 2, 3, 4)
#     nom_operateur  TEXT NOT NULL UNIQUE,       -- MTN CI, Orange Money, Moov Africa, Wave
#     type_operateur TEXT,                       -- Telecoms ou Fintech
#     actif          BOOLEAN DEFAULT TRUE        -- TRUE si l'opérateur est encore en activité
# );

# -- ------------------------------------------------------------
# -- DIMENSION 2 : Zone géographique
# -- Décrit les zones d'Abidjan et les villes de l'intérieur CI
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS dim_zone (
#     id_zone    SERIAL PRIMARY KEY,             -- identifiant auto-incrémenté
#     nom_zone   TEXT NOT NULL UNIQUE,           -- ex : Abidjan-Cocody, Bouaké, Korhogo
#     region     TEXT,                           -- ex : Abidjan, Centre, Nord, Ouest
#     type_zone  TEXT,                           -- Quartier Abidjan ou Ville intérieur
#     population INTEGER                         -- population estimée de la zone
# );

# -- ------------------------------------------------------------
# -- DIMENSION 3 : Type d'opération
# -- Décrit les 5 types d'opérations Mobile Money disponibles
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS dim_type_operation (
#     id_type        SERIAL PRIMARY KEY,         -- identifiant auto-incrémenté
#     nom_type       TEXT NOT NULL UNIQUE,       -- Transfert, Dépôt, Retrait, Paiement, Recharge
#     categorie      TEXT,                       -- Envoi de fonds, Entrée fonds, Sortie fonds...
#     taux_frais_max NUMERIC(4,2)               -- taux de frais maximum appliqué (en %)
# );

# -- ------------------------------------------------------------
# -- DIMENSION 4 : Calendrier
# -- Permet d'analyser les tendances temporelles (mois, trimestre)
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS dim_date (
#     id_date       SERIAL PRIMARY KEY,          -- identifiant auto-incrémenté
#     date_complete DATE NOT NULL UNIQUE,        -- ex : 2024-01-15
#     annee         INTEGER,                     -- ex : 2024
#     mois          INTEGER,                     -- 1 à 12
#     nom_mois      TEXT,                        -- Janvier, Février...
#     trimestre     INTEGER,                     -- 1 à 4
#     semaine       INTEGER,                     -- 1 à 52
#     jour          INTEGER,                     -- 1 à 31
#     nom_jour      TEXT,                        -- Lundi, Mardi...
#     est_weekend   BOOLEAN                      -- TRUE si samedi ou dimanche
# );


# -- ============================================================
# -- PARTIE 3 : TABLE DE FAITS
# -- Centre du schéma en étoile — contient les mesures
# -- et les clés étrangères vers toutes les dimensions
# -- ============================================================

# CREATE TABLE IF NOT EXISTS faits_transactions (
#     id_transaction    TEXT PRIMARY KEY,        -- identifiant unique de la transaction

#     -- Clés étrangères vers les dimensions (le cœur du schéma en étoile)
#     id_operateur      INTEGER REFERENCES dim_operateur(id_operateur),
#     id_zone_exp       INTEGER REFERENCES dim_zone(id_zone),       -- zone expéditeur
#     id_zone_ben       INTEGER REFERENCES dim_zone(id_zone),       -- zone bénéficiaire
#     id_type           INTEGER REFERENCES dim_type_operation(id_type),
#     id_date           INTEGER REFERENCES dim_date(id_date),

#     -- Mesures (les chiffres à analyser)
#     montant_fcfa      BIGINT NOT NULL,         -- montant brut de la transaction
#     frais_fcfa        INTEGER DEFAULT 0,       -- frais prélevés par l'opérateur
#     montant_net_fcfa  BIGINT,                  -- montant net reçu par le bénéficiaire
#     taux_frais_pct    NUMERIC(5,2),            -- pourcentage de frais sur le montant
#     heure             INTEGER,                 -- heure de la transaction (0 à 23)
#     statut            TEXT,                    -- Succès, Échec, En attente
#     inter_ville       INTEGER DEFAULT 0,       -- 1 = transaction entre deux villes différentes
#     id_agent          TEXT                     -- agent ayant traité la transaction
# );


# -- ============================================================
# -- PARTIE 4 : TABLES D'AGRÉGATION
# -- Résumés précalculés pour accélérer les analyses
# -- et alimenter directement le tableau de bord
# -- ============================================================

# -- ------------------------------------------------------------
# -- AGRÉGATION 1 : Performance par opérateur
# -- Résume le volume, les frais et le taux de succès par opérateur
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS agg_operateur (
#     operateur         TEXT PRIMARY KEY,        -- nom de l'opérateur
#     nb_transactions   INTEGER,                 -- nombre total de transactions
#     volume_total      BIGINT,                  -- volume total en FCFA
#     frais_total       BIGINT,                  -- frais totaux collectés en FCFA
#     montant_moyen     NUMERIC(12,2),           -- montant moyen par transaction
#     taux_succes_pct   NUMERIC(5,2)            -- pourcentage de transactions réussies
# );

# -- ------------------------------------------------------------
# -- AGRÉGATION 2 : Performance par type d'opération
# -- Résume les transactions par type (Dépôt, Retrait, etc.)
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS agg_type_operation (
#     type_operation    TEXT PRIMARY KEY,        -- nom du type d'opération
#     nb_transactions   INTEGER,                 -- nombre de transactions de ce type
#     volume_total      BIGINT,                  -- volume total en FCFA
#     frais_total       BIGINT,                  -- frais totaux collectés
#     montant_moyen     NUMERIC(12,2)            -- montant moyen par transaction
# );

# -- ------------------------------------------------------------
# -- AGRÉGATION 3 : Performance par zone géographique
# -- Résume les transactions par zone expéditeur
# -- ------------------------------------------------------------
# CREATE TABLE IF NOT EXISTS agg_zone (
#     zone              TEXT PRIMARY KEY,        -- nom de la zone géographique
#     nb_transactions   INTEGER,                 -- nombre de transactions émises
#     volume_total      BIGINT,                  -- volume total émis en FCFA
#     frais_total       BIGINT,                  -- frais totaux collectés
#     montant_moyen     NUMERIC(12,2)            -- montant moyen par transaction
# );


# -- ============================================================
# -- VÉRIFICATION FINALE
# -- Lister toutes les tables créées dans le schéma public
# -- ============================================================
# SELECT
#     table_name,
#     pg_size_pretty(pg_total_relation_size(quote_ident(table_name))) AS taille
# FROM information_schema.tables
# WHERE table_schema = 'public'
# ORDER BY table_name;
# """

# with open('/content/mobile-money-pipeline-ci/sql/schema_mobile_money.sql', 'w', encoding='utf-8') as f:
#     f.write(sql_content.strip())

# print("✅ Fichier SQL créé : sql/schema_mobile_money.sql")